In [1]:
import argparse
import csv
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager


BASE = "https://www.cbo.gov"
MAJOR_URL = "https://www.cbo.gov/about/products/major-recurring-reports"


# ---------- small utils ----------
def dedupe_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def clean_filename(s, max_len=180):
    s = unquote(s)
    s = re.sub(r"[^\w\-. ()]+", "_", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:max_len] or "file"


def publication_id(url):
    m = re.search(r"/publication/(\d+)", url)
    return m.group(1) if m else "unknown"


# ---------- selenium ----------
def selenium_driver(headless=False, user_data_dir="cbo_selenium_profile", download_dir="downloads"):
    """
    Configure Chrome to download files to download_dir.
    NOTE: if you re-use an existing profile, some prefs may be overridden by the profile.
    This script also uses CDP setDownloadBehavior after launch.
    """
    download_dir = str(Path(download_dir).resolve())

    opts = ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")

    # Persistent profile: keeps DataDome solved state/cookies
    opts.add_argument(f"--user-data-dir={str(Path(user_data_dir).resolve())}")

    # Attempt to force PDFs to download instead of opening in viewer
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True,  # key for PDFs
    }
    opts.add_experimental_option("prefs", prefs)

    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--start-maximized")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)

    # Ensure downloads are allowed to our path (works even if prefs partially ignored)
    try:
        driver.execute_cdp_cmd("Page.setDownloadBehavior", {
            "behavior": "allow",
            "downloadPath": download_dir
        })
    except Exception:
        pass

    return driver


def wait_for_budget_outlook_section(driver, timeout=600):
    wait = WebDriverWait(driver, timeout)

    wait.until(EC.presence_of_element_located(
        (By.XPATH, "//h4[normalize-space(.)='Budget and Economic Outlook and Updates']")
    ))

    wait.until(EC.presence_of_element_located(
        (By.XPATH,
         "//h4[normalize-space(.)='Budget and Economic Outlook and Updates']"
         "/ancestor::div[contains(@class,'view-major-recurring-reports')]"
         "//a[contains(@href,'/publication/')]")
    ))


def wait_for_publication_doc_links(driver, timeout=40):
    wait = WebDriverWait(driver, timeout)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1, #page-title")))

    def any_doc_link_present(drv):
        for a in drv.find_elements(By.CSS_SELECTOR, "a[href]"):
            href = (a.get_attribute("href") or "").lower()
            text = (a.text or "").strip().lower()
            if ".pdf" in href or "/sites/default/files/" in href or "system/files" in href or text == "view document":
                return True
        return False

    wait.until(any_doc_link_present)


# ---------- parsing ----------
def extract_budget_outlook_publication_links_from_html(html):
    soup = BeautifulSoup(html, "html.parser")

    target_h4 = None
    for h4 in soup.find_all("h4"):
        if h4.get_text(" ", strip=True) == "Budget and Economic Outlook and Updates":
            target_h4 = h4
            break
    if not target_h4:
        return []

    view = target_h4.find_parent(
        lambda tag: tag.name == "div"
        and "view-major-recurring-reports" in tag.get("class", [])
    )
    if not view:
        return []

    links = []
    for a in view.select('a[href*="/publication/"]'):
        href = a.get("href")
        if href:
            links.append(urljoin(BASE, href))

    return dedupe_preserve_order(links)


def extract_title(pub_html):
    soup = BeautifulSoup(pub_html, "html.parser")
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(" ", strip=True)
    t = soup.find("title")
    return t.get_text(" ", strip=True) if t else ""


def extract_pub_date_yyyymmdd(pub_html):
    soup = BeautifulSoup(pub_html, "html.parser")
    t = soup.find("time", attrs={"datetime": True})
    if not t:
        return ""
    dt = (t.get("datetime") or "").strip()  # e.g. 2000-01-01T12:00:00Z
    m = re.match(r"^(\d{4})-(\d{2})-(\d{2})", dt)
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}" if m else ""


def extract_pdf_links(pub_html, pub_url):
    soup = BeautifulSoup(pub_html, "html.parser")
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        text = (a.get_text(" ", strip=True) or "").lower()
        href_l = href.lower()

        looks_like_doc = (
            ".pdf" in href_l
            or "/sites/default/files/" in href_l
            or ("system/files" in href_l)  # includes /system/files?file=...
            or text == "view document"
        )
        if not looks_like_doc:
            continue

        full = urljoin(pub_url, href)
        if urlparse(full).netloc.lower().endswith("cbo.gov"):
            links.append(full)

    links = dedupe_preserve_order(links)
    pdf_only = [u for u in links if ".pdf" in u.lower()]
    return pdf_only if pdf_only else links


def choose_one_main_pdf(pdf_links):
    if not pdf_links:
        return None

    bad_terms = [
        "by-the-numbers", "by_the_numbers", "by the numbers",
        "executive", "handout", "visual", "at-a-glance", "glance",
        "notes", "chapter", "appendix", "about-this-document", "cover",
    ]
    good_terms = ["outlook", "economy", "econ-outlook", "economic-outlook", "budget-outlook"]

    def score(url):
        u = url.lower()
        s = 0
        for t in good_terms:
            if t in u:
                s += 50
        if "outlook" in u:
            s += 40
        if "system/files?" in u and "file=" in u:
            s -= 5  # slight penalty, but still ok
        for t in bad_terms:
            if t in u:
                s -= 200
        if ".pdf" in u:
            s += 10
        return s

    return sorted(pdf_links, key=score, reverse=True)[0]


# ---------- downloading via browser ----------
def wait_for_download_to_finish(download_dir: Path, before_set: set, timeout=180):
    """
    Wait until a new file appears in download_dir and no .crdownload remains for it.
    Returns the final Path.
    """
    start = time.time()
    while time.time() - start < timeout:
        current = set(p.name for p in download_dir.iterdir() if p.is_file())
        new_files = list(current - before_set)

        # ignore temporary/incomplete
        new_files = [f for f in new_files if not f.endswith(".crdownload")]

        # also ensure no .crdownload exists (download still running)
        crs = [p for p in download_dir.glob("*.crdownload")]
        if new_files and not crs:
            # pick newest by mtime
            candidates = [download_dir / f for f in new_files]
            candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            return candidates[0]

        time.sleep(1.0)

    return None


def download_one_pdf_via_browser(driver, pdf_url, out_dir, final_filename, download_timeout=240):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    before = set(p.name for p in out_dir.iterdir() if p.is_file())

    # Trigger download
    driver.get(pdf_url)

    downloaded = wait_for_download_to_finish(out_dir, before, timeout=download_timeout)
    if not downloaded:
        return None, "download_timeout"

    final_path = out_dir / final_filename
    if final_path.exists():
        # avoid overwrite
        stem = final_path.stem
        suffix = final_path.suffix
        i = 2
        while True:
            alt = out_dir / f"{stem}__{i}{suffix}"
            if not alt.exists():
                final_path = alt
                break
            i += 1

    downloaded.rename(final_path)
    return final_path, "downloaded"


# ---------- main ----------
def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default="cbo_budget_economic_outlook_mainpdfs", help="Output directory")
    parser.add_argument("--headless", action="store_true", help="Headless (not recommended)")
    parser.add_argument("--profile", default="cbo_selenium_profile", help="Chrome user-data-dir folder")
    parser.add_argument("--delay", type=float, default=0.6, help="Delay between publication pages")
    parser.add_argument("--timeout", type=int, default=600, help="Wait for slider/challenge on listing page")
    parser.add_argument("--pub-timeout", type=int, default=40, help="Wait for doc links on publication pages")
    parser.add_argument("--download-timeout", type=int, default=240, help="Wait for each PDF download")
    args = parser.parse_args([] if argv is None else argv)

    out_dir = Path(args.out).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.csv"

    driver = selenium_driver(
        headless=args.headless,
        user_data_dir=args.profile,
        download_dir=str(out_dir),
    )

    rows = []
    try:
        print(f"Opening listing page: {MAJOR_URL}")
        driver.get(MAJOR_URL)

        print("\nWaiting for the Budget and Economic Outlook section to become available.")
        print("If a slider appears, solve it in the Chrome window.\n")
        wait_for_budget_outlook_section(driver, timeout=args.timeout)

        main_html = driver.page_source
        publication_links = extract_budget_outlook_publication_links_from_html(main_html)
        print(f"\nFound {len(publication_links)} publication pages.\n")

        for n, pub_url in enumerate(publication_links, start=1):
            print("=" * 80)
            print(f"[{n}/{len(publication_links)}] {pub_url}")

            driver.get(pub_url)
            try:
                wait_for_publication_doc_links(driver, timeout=args.pub_timeout)
            except Exception:
                pass

            pub_html = driver.page_source
            title = extract_title(pub_html)
            pub_date = extract_pub_date_yyyymmdd(pub_html)

            pdf_links = extract_pdf_links(pub_html, pub_url)
            chosen = choose_one_main_pdf(pdf_links)

            if not chosen:
                print("  No PDF links found.")
                rows.append({
                    "publication_url": pub_url,
                    "publication_id": publication_id(pub_url),
                    "pub_date": pub_date,
                    "title": title,
                    "chosen_pdf_url": "",
                    "saved_as": "",
                    "status": "no_pdf_found",
                })
            else:
                print(f"  Chosen main PDF: {chosen}")

                date_part = pub_date if pub_date else "unknown-date"
                pid = publication_id(pub_url)
                title_part = clean_filename(title, max_len=120)
                final_name = clean_filename(f"{date_part}__{pid}__{title_part}.pdf", max_len=220)

                saved_path, status = download_one_pdf_via_browser(
                    driver=driver,
                    pdf_url=chosen,
                    out_dir=out_dir,
                    final_filename=final_name,
                    download_timeout=args.download_timeout,
                )

                if status == "downloaded":
                    print(f"  Saved: {saved_path.name}")
                else:
                    print(f"  Download failed: {status}")

                rows.append({
                    "publication_url": pub_url,
                    "publication_id": pid,
                    "pub_date": pub_date,
                    "title": title,
                    "chosen_pdf_url": chosen,
                    "saved_as": str(saved_path) if saved_path else "",
                    "status": status,
                })

            time.sleep(args.delay)

        with manifest_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(
                f,
                fieldnames=["publication_url","publication_id","pub_date","title","chosen_pdf_url","saved_as","status"]
            )
            w.writeheader()
            w.writerows(rows)

        print(f"\nDone.\nManifest: {manifest_path}\nOutput: {out_dir}")

    finally:
        driver.quit()

In [2]:
main(["--profile", "cbo_selenium_profile_fresh"])

Opening listing page: https://www.cbo.gov/about/products/major-recurring-reports

Waiting for the Budget and Economic Outlook section to become available.
If a slider appears, solve it in the Chrome window.


Found 70 publication pages.

[1/70] https://www.cbo.gov/publication/61882
  Chosen main PDF: https://www.cbo.gov/system/files/2026-02/61882-Outlook-2026.pdf
  Saved: 2026-02-11__61882__The Budget and Economic Outlook_ 2026 to 2036.pdf
[2/70] https://www.cbo.gov/publication/61831
  Chosen main PDF: https://www.cbo.gov/system/files/2026-01/61831-Economy.pdf
  Saved: 2026-01-08__61831__CBO_s Current View of the Economy From 2026 to 2028.pdf
[3/70] https://www.cbo.gov/publication/61236
  Chosen main PDF: https://www.cbo.gov/system/files/2025-09/61236-Economy.pdf
  Saved: 2025-09-12__61236__CBO_s Current View of the Economy From 2025 to 2028.pdf
[4/70] https://www.cbo.gov/publication/61135
  Chosen main PDF: https://www.cbo.gov/system/files/2025-01/61135-econ-outlook.pdf
  Saved: 2025-